# Import YOLO model

In [ ]:
from ultralytics import YOLO

# 1. Load the latest NMS-free model (Nano version is best for speed)
model = YOLO("yolo11n.pt") 

# 2. Run detection on an image, video, or folder
results = model("0000168_01491_d_0000170.jpg")

# 3. View the results
for result in results:
    result.show()  # Opens a window with detected objects
    result.save(filename="result.jpg")  # Saves the output to disk

# Obtain bounding boxes

import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model("0000168_01491_d_0000170.jpg")

for result in results:
    for i, box in enumerate(result.boxes):
        # 1. Get coordinates (xmin, ymin, xmax, ymax)
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # 2. Crop the image using numpy slicing [y1:y2, x1:x2]
        crop = result.orig_img[y1:y2, x1:x2]
        
        # 3. Save the specific crop
        filename = f"crop_object_{i}.jpg"
        cv2.imwrite(filename, crop)
        print(f"Saved: {filename}")

# Expand coordiates of bounding boxes

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model("0000168_01491_d_0000170.jpg")

# Define the padding amount
padding = 20

for result in results:
    # Get original image dimensions for boundary checking
    h, w, _ = result.orig_img.shape
    
    for i, box in enumerate(result.boxes):
        # 1. Get original coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # 2. Expand coordinates with boundary protection
        # max(0, ...) ensures we don't go below pixel 0
        # min(limit, ...) ensures we don't exceed image width/height
        x1_ext = max(0, x1 - padding)
        y1_ext = max(0, y1 - padding)
        x2_ext = min(w, x2 + padding)
        y2_ext = min(h, y2 + padding)
        
        # 3. Crop using the expanded coordinates
        crop = result.orig_img[y1_ext:y2_ext, x1_ext:x2_ext]
        
        # 4. Save the expanded crop
        filename = f"expanded_crop_{i}.jpg"
        cv2.imwrite(filename, crop)
        print(f"Saved expanded crop: {filename} (Size: {x2_ext-x1_ext}x{y2_ext-y1_ext})")

# Custom backgound blurring

In [ ]:
import cv2
from ultralytics import YOLO

# 1. Load the model and image
model = YOLO("yolo11n.pt") 
image_path = "0000077_00075_d_0000001.jpg"
results = model(image_path)
padding = 20

# 2. Compress the ENTIRE image first to 35% quality
# We encode it to a buffer and decode it back to get the 'compressed' version in memory
img_orig = cv2.imread(image_path)
encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 35]
_, buffer = cv2.imencode('.jpg', img_orig, encode_param)
img_compressed = cv2.imdecode(buffer, 1)

for result in results:
    h, w, _ = img_orig.shape
    
    for i, box in enumerate(result.boxes):
        # 3. Get coordinates for the high-quality crop (with padding)
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1_ext = max(0, x1 - padding)
        y1_ext = max(0, y1 - padding)
        x2_ext = min(w, x2 + padding)
        y2_ext = min(h, y2 + padding)
        
        # 4. Extract the High-Quality crop from the ORIGINAL uncompressed image
        high_res_crop = img_orig[y1_ext:y2_ext, x1_ext:x2_ext]
        
        # 5. Paste the High-Quality crop onto the Compressed background
        img_compressed[y1_ext:y2_ext, x1_ext:x2_ext] = high_res_crop
        
        # 6. Save the final hybrid image
        filename = f"hybrid_compressed_{i}.jpg"
        # Note: Save at 100% here so we don't re-compress our high-res patch!
        cv2.imwrite(filename, img_compressed, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
        
        print(f"Saved hybrid image: {filename}")

# Load MobileNet and classify images

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import mobilenet_v3
from tensorflow.keras.preprocessing import image
import numpy as np

# 1. Load the pre-trained MobileNetV3 (Large or Small)
model = tf.keras.applications.MobileNetV3Large(weights='imagenet')

# 2. Load and preprocess your image (MobileNet expects 224x224 pixels)

img_path = '0000168_01491_d_0000170.jpg'
#img_path = 'hybrid_compressed_2 0000168_03306_d_0000174.jpg'
img = image.load_img(img_path, target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = mobilenet_v3.preprocess_input(x)

# 3. Predict the class
preds = model.predict(x)

# 4. Decode results into human-readable labels
print('Predicted:', mobilenet_v3.decode_predictions(preds, top=3)[0])